# Extreme Climate Events: 04_Dashboard

## Data Loading & Exploration

### Concept: Data Aggregation for Visualization

When building dashboards, we work with **aggregated** data (summarized by dimension) rather than raw event-level data. This makes visualizations clearer and faster.

Key idea: Raw data has thousands of individual events → We group by State, Year, Month, Event Type to create meaningful trends.

In [9]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

In [10]:
# Load the engineered features dataset
df = pd.read_csv('../data/processed/features.csv')

print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())

Dataset Shape: (1520485, 12)

First few rows:
   EVENT_ID          STATE  YEAR  MONTH  QUARTER               EVENT_TYPE  \
0   5165377        FLORIDA  2000     12        4  Extreme Cold/Wind Chill   
1   5165378        FLORIDA  2000     12        4  Extreme Cold/Wind Chill   
2   5165379        FLORIDA  2000     12        4  Extreme Cold/Wind Chill   
3   5165449  WEST VIRGINIA  2000     12        4             Winter Storm   
4   5172568    MISSISSIPPI  2000      8        3        Thunderstorm Wind   

   ANNUAL_EVENT_COUNT  ROLLING_AVG_3Y  IS_PEAK_SEASON  TOTAL_DAMAGE  \
0              1244.0          1244.0               0           0.0   
1              1244.0          1244.0               0           0.0   
2              1244.0          1244.0               0           0.0   
3              1123.0          1123.0               0           0.0   
4               934.0           934.0               0        2000.0   

   TOTAL_DEATHS  SEVERITY_INDEX  
0             0         0.0000

In [11]:
# Check unique values in key columns
print("\n" + "="*60)
print("KEY DIMENSIONS")
print("="*60)
print(f"\nUnique States: {df['STATE'].nunique()}")
print(f"States: {sorted(df['STATE'].dropna().unique())}")
print(f"\nYear Range: {df['YEAR'].min()} - {df['YEAR'].max()}")
print(f"Unique Event Types: {df['EVENT_TYPE'].nunique()}")
print(f"\nTop 10 Event Types:")
print(df['EVENT_TYPE'].value_counts().head(10))


KEY DIMENSIONS

Unique States: 69
States: ['ALABAMA', 'ALASKA', 'AMERICAN SAMOA', 'ARIZONA', 'ARKANSAS', 'ATLANTIC NORTH', 'ATLANTIC SOUTH', 'CALIFORNIA', 'COLORADO', 'CONNECTICUT', 'DELAWARE', 'DISTRICT OF COLUMBIA', 'E PACIFIC', 'FLORIDA', 'GEORGIA', 'GUAM', 'GUAM WATERS', 'GULF OF ALASKA', 'GULF OF MEXICO', 'HAWAII', 'HAWAII WATERS', 'IDAHO', 'ILLINOIS', 'INDIANA', 'IOWA', 'KANSAS', 'KENTUCKY', 'LAKE ERIE', 'LAKE HURON', 'LAKE MICHIGAN', 'LAKE ONTARIO', 'LAKE ST CLAIR', 'LAKE SUPERIOR', 'LOUISIANA', 'MAINE', 'MARYLAND', 'MASSACHUSETTS', 'MICHIGAN', 'MINNESOTA', 'MISSISSIPPI', 'MISSOURI', 'MONTANA', 'NEBRASKA', 'NEVADA', 'NEW HAMPSHIRE', 'NEW JERSEY', 'NEW MEXICO', 'NEW YORK', 'NORTH CAROLINA', 'NORTH DAKOTA', 'OHIO', 'OKLAHOMA', 'OREGON', 'PENNSYLVANIA', 'PUERTO RICO', 'RHODE ISLAND', 'SOUTH CAROLINA', 'SOUTH DAKOTA', 'ST LAWRENCE R', 'TENNESSEE', 'TEXAS', 'UTAH', 'VERMONT', 'VIRGIN ISLANDS', 'VIRGINIA', 'WASHINGTON', 'WEST VIRGINIA', 'WISCONSIN', 'WYOMING']

Year Range: 2000 - 202

In [12]:
# Examine data types and missing values
print("\n" + "="*60)
print("DATA STRUCTURE")
print("="*60)
print("\nColumn Data Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nBasic Statistics:")
print(df.describe())


DATA STRUCTURE

Column Data Types:
EVENT_ID                int64
STATE                     str
YEAR                    int64
MONTH                   int64
QUARTER                 int64
EVENT_TYPE                str
ANNUAL_EVENT_COUNT    float64
ROLLING_AVG_3Y        float64
IS_PEAK_SEASON          int64
TOTAL_DAMAGE          float64
TOTAL_DEATHS            int64
SEVERITY_INDEX        float64
dtype: object

Missing Values:
EVENT_ID              0
STATE                 1
YEAR                  0
MONTH                 0
QUARTER               0
EVENT_TYPE            0
ANNUAL_EVENT_COUNT    1
ROLLING_AVG_3Y        1
IS_PEAK_SEASON        0
TOTAL_DAMAGE          0
TOTAL_DEATHS          0
SEVERITY_INDEX        0
dtype: int64

Basic Statistics:
           EVENT_ID          YEAR         MONTH       QUARTER  \
count  1.520485e+06  1.520485e+06  1.520485e+06  1.520485e+06   
mean   1.733196e+06  2.012581e+03  6.035241e+00  2.348643e+00   
std    2.036833e+06  7.142479e+00  3.049698e+00  9.842863e

## 5-Year Forecast Generation

### Forecasting Strategy

To predict 2025-2029, we:
1. **Retrain the model** on all available data (no train/test split) to capture latest patterns
2. **Create synthetic future data** with same feature columns
3. **Make predictions** for each state × year combination
4. **Ensemble approach**: Average predictions from Random Forest & XGBoost for robustness

**Key Insight**: We're extrapolating beyond training data (2000-2024), so confidence decreases with distance. Use forecasts as trends, not absolute values.

In [13]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

# Prepare data for model training (use all available data to capture latest trends)
df_model = df.copy()
df_model = df_model.dropna(subset=['SEVERITY_INDEX'])

# Encode categorical variables (IMPORTANT: Use same encoding as training data)
le_state = LabelEncoder()
le_event = LabelEncoder()

df_model['STATE_ENCODED'] = le_state.fit_transform(df_model['STATE'])
df_model['EVENT_TYPE_ENCODED'] = le_event.fit_transform(df_model['EVENT_TYPE'])

# Define feature columns (MUST match notebook 03)
feature_columns = [
    'STATE_ENCODED', 'YEAR', 'MONTH', 'QUARTER', 'EVENT_TYPE_ENCODED',
    'ROLLING_AVG_3Y', 'IS_PEAK_SEASON', 'TOTAL_DAMAGE', 'TOTAL_DEATHS'
]

X = df_model[feature_columns].fillna(0)
y = df_model['SEVERITY_INDEX']

print("Training data shape:", X.shape)
print("Feature columns:", feature_columns)
print("\nModels will be trained on ALL historical data (2000-2024)")


Training data shape: (1520485, 9)
Feature columns: ['STATE_ENCODED', 'YEAR', 'MONTH', 'QUARTER', 'EVENT_TYPE_ENCODED', 'ROLLING_AVG_3Y', 'IS_PEAK_SEASON', 'TOTAL_DAMAGE', 'TOTAL_DEATHS']

Models will be trained on ALL historical data (2000-2024)


In [14]:
# Train Random Forest on all available data
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X, y)

# Train XGBoost on all available data
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X, y)

print("✓ Random Forest trained")
print("✓ XGBoost trained")
print("\nModels are ready for forecasting!")


✓ Random Forest trained
✓ XGBoost trained

Models are ready for forecasting!


In [15]:
# Create future data for 2025-2029
# Strategy: For each future year, create a row for each state with reasonable assumptions

# Get the average values of numeric features from historical data
avg_rolling_avg = df_model['ROLLING_AVG_3Y'].mean()
avg_damage = df_model['TOTAL_DAMAGE'].mean()
avg_deaths = df_model['TOTAL_DEATHS'].mean()

# Most common event type and peak season occurrence
most_common_event = df_model['EVENT_TYPE'].mode()[0]
most_common_month = df_model['MONTH'].mode()[0]
peak_season_prob = df_model['IS_PEAK_SEASON'].mean()

print("Future Data Assumptions:")
print(f"  Average rolling average (3Y): {avg_rolling_avg:.2f}")
print(f"  Average damage: {avg_damage:.2f}")
print(f"  Average deaths: {avg_deaths:.2f}")
print(f"  Most common event type: {most_common_event}")
print(f"  Most common month: {most_common_month}")

# Build forecast dataframe for 2025-2029
forecast_years = [2025, 2026, 2027, 2028, 2029]
states = df_model['STATE'].unique()

forecast_data = []

for year in forecast_years:
    for state in states:
        # Get encoded values for this state and event
        state_encoded = le_state.transform([state])[0]
        event_encoded = le_event.transform([most_common_event])[0]
        
        # Quarter based on month
        quarter = (most_common_month - 1) // 3 + 1
        
        # Determine peak season (assume same probability as historical)
        is_peak = 1 if np.random.random() < peak_season_prob else 0
        
        forecast_data.append({
            'STATE': state,
            'YEAR': year,
            'MONTH': most_common_month,
            'QUARTER': quarter,
            'EVENT_TYPE': most_common_event,
            'STATE_ENCODED': state_encoded,
            'EVENT_TYPE_ENCODED': event_encoded,
            'ROLLING_AVG_3Y': avg_rolling_avg,
            'IS_PEAK_SEASON': is_peak,
            'TOTAL_DAMAGE': avg_damage,
            'TOTAL_DEATHS': avg_deaths
        })

df_forecast = pd.DataFrame(forecast_data)
print(f"\nForecast data shape: {df_forecast.shape}")
print(f"Years: {df_forecast['YEAR'].unique()}")
print(f"\nSample forecast rows:")
print(df_forecast.head(10))


Future Data Assumptions:
  Average rolling average (3Y): 1685.75
  Average damage: 382791.14
  Average deaths: 0.01
  Most common event type: Thunderstorm Wind
  Most common month: 6

Forecast data shape: (350, 11)
Years: [2025 2026 2027 2028 2029]

Sample forecast rows:
            STATE  YEAR  MONTH  QUARTER         EVENT_TYPE  STATE_ENCODED  \
0         FLORIDA  2025      6        2  Thunderstorm Wind             13   
1   WEST VIRGINIA  2025      6        2  Thunderstorm Wind             66   
2     MISSISSIPPI  2025      6        2  Thunderstorm Wind             39   
3           MAINE  2025      6        2  Thunderstorm Wind             34   
4     CONNECTICUT  2025      6        2  Thunderstorm Wind              9   
5         GEORGIA  2025      6        2  Thunderstorm Wind             14   
6  NORTH CAROLINA  2025      6        2  Thunderstorm Wind             48   
7         ARIZONA  2025      6        2  Thunderstorm Wind              3   
8           TEXAS  2025      6     

In [16]:
# Make predictions using both models
X_forecast = df_forecast[feature_columns].fillna(0)

# Get predictions from both models
rf_predictions = rf_model.predict(X_forecast)
xgb_predictions = xgb_model.predict(X_forecast)

# Ensemble: Average predictions for robustness
# This reduces variance and combines strengths of both models
ensemble_predictions = (rf_predictions + xgb_predictions) / 2

df_forecast['PREDICTED_SEVERITY_RF'] = rf_predictions
df_forecast['PREDICTED_SEVERITY_XGB'] = xgb_predictions
df_forecast['PREDICTED_SEVERITY_ENSEMBLE'] = ensemble_predictions

print("=" * 70)
print("FORECAST PREDICTIONS GENERATED")
print("=" * 70)
print(f"\nPredictions shape: {len(df_forecast)}")
print(f"\nEnsemble Severity Index Statistics (2025-2029):")
print(df_forecast['PREDICTED_SEVERITY_ENSEMBLE'].describe())

print("\n\nSample Forecast Results:")
print(df_forecast[['STATE', 'YEAR', 'PREDICTED_SEVERITY_RF', 'PREDICTED_SEVERITY_XGB', 'PREDICTED_SEVERITY_ENSEMBLE']].head(15))


FORECAST PREDICTIONS GENERATED

Predictions shape: 350

Ensemble Severity Index Statistics (2025-2029):
count    350.000000
mean       0.325666
std        0.001733
min        0.317180
25%        0.325453
50%        0.325594
75%        0.326345
max        0.328844
Name: PREDICTED_SEVERITY_ENSEMBLE, dtype: float64


Sample Forecast Results:
             STATE  YEAR  PREDICTED_SEVERITY_RF  PREDICTED_SEVERITY_XGB  \
0          FLORIDA  2025               0.326687                0.324138   
1    WEST VIRGINIA  2025               0.326689                0.327312   
2      MISSISSIPPI  2025               0.326687                0.326387   
3            MAINE  2025               0.326687                0.324636   
4      CONNECTICUT  2025               0.326687                0.324138   
5          GEORGIA  2025               0.326687                0.325357   
6   NORTH CAROLINA  2025               0.326685                0.325179   
7          ARIZONA  2025               0.326687            

In [17]:
# Aggregate forecast by State and Year (for cleaner visualization)
forecast_summary = df_forecast.groupby(['STATE', 'YEAR']).agg({
    'PREDICTED_SEVERITY_ENSEMBLE': 'mean',
    'PREDICTED_SEVERITY_RF': 'mean',
    'PREDICTED_SEVERITY_XGB': 'mean'
}).reset_index()

forecast_summary.columns = ['STATE', 'YEAR', 'SEVERITY_FORECAST', 'SEVERITY_FORECAST_RF', 'SEVERITY_FORECAST_XGB']

print("\n" + "=" * 70)
print("FORECAST SUMMARY (by State × Year)")
print("=" * 70)
print(f"Shape: {forecast_summary.shape}")
print(f"\nTop 10 States with Highest Predicted Severity (2025-2029 average):")
top_states = df_forecast.groupby('STATE')['PREDICTED_SEVERITY_ENSEMBLE'].mean().nlargest(10)
print(top_states)

print(f"\n\nForecast Summary Sample:")
print(forecast_summary.head(15))

# Save for next step (visualization)
print("\n✓ Forecast data ready for visualization step")



FORECAST SUMMARY (by State × Year)
Shape: (345, 5)

Top 10 States with Highest Predicted Severity (2025-2029 average):
STATE
VERMONT           0.328844
VIRGINIA          0.328844
VIRGIN ISLANDS    0.328795
WASHINGTON        0.328485
ST LAWRENCE R     0.328044
NEW YORK          0.327236
WEST VIRGINIA     0.327148
TENNESSEE         0.327057
TEXAS             0.327036
UTAH              0.327008
Name: PREDICTED_SEVERITY_ENSEMBLE, dtype: float64


Forecast Summary Sample:
             STATE  YEAR  SEVERITY_FORECAST  SEVERITY_FORECAST_RF  \
0          ALABAMA  2025           0.326266              0.326687   
1          ALABAMA  2026           0.326266              0.326687   
2          ALABAMA  2027           0.326321              0.326687   
3          ALABAMA  2028           0.326321              0.326687   
4          ALABAMA  2029           0.326321              0.326687   
5           ALASKA  2025           0.326345              0.326687   
6           ALASKA  2026           0.326345 

## Individual Visualizations

### Visualization Strategy

We'll create 5 complementary charts:
1. **Time Series (Historical)** — Show trends in severity from 2000-2024
2. **Forecast Chart** — Project severity forward to 2025-2029
3. **Top States Bar Chart** — Identify high-risk states
4. **Event Type Breakdown** — Which events are most damaging
5. **Seasonal Heatmap** — Which months/states are most dangerous

**Key Design Principle**: Each chart answers a specific question. Users can drill from "what states are affected?" → "when?" → "by which events?"

In [ ]:
# Chart 1: Historical Severity Trend (2000-2024)
# Aggregate severity by year from historical data

historical_trend = df.groupby('YEAR').agg({
    'SEVERITY_INDEX': ['mean', 'sum', 'count']
}).reset_index()

historical_trend.columns = ['YEAR', 'AVG_SEVERITY', 'TOTAL_SEVERITY', 'EVENT_COUNT']

fig1 = go.Figure()

# Add average severity line
fig1.add_trace(go.Scatter(
    x=historical_trend['YEAR'],
    y=historical_trend['AVG_SEVERITY'],
    name='Average Severity',
    mode='lines+markers',
    line=dict(color='#FF6B6B', width=3),
    marker=dict(size=6),
    hovertemplate='<b>%{x}</b><br>Avg Severity: %{y:.3f}<extra></extra>'
))

fig1.update_layout(
    title='<b>Historical Severity Trend (2000-2024)</b><br><sub>Annual average extreme weather severity index</sub>',
    xaxis_title='Year',
    yaxis_title='Average Severity Index',
    template='plotly_white',
    height=500,
    hovermode='x unified',
    font=dict(size=12)
)

fig1.show()


In [ ]:
# Chart 2: Historical vs Forecast Comparison
# Combine historical data with forecast data for comparison

# Historical aggregate by year
historical_yearly = df.groupby('YEAR')['SEVERITY_INDEX'].mean().reset_index()
historical_yearly.columns = ['YEAR', 'SEVERITY']
historical_yearly['TYPE'] = 'Historical'

# Forecast aggregate by year
forecast_yearly = forecast_summary.groupby('YEAR')['SEVERITY_FORECAST'].mean().reset_index()
forecast_yearly.columns = ['YEAR', 'SEVERITY']
forecast_yearly['TYPE'] = 'Forecast'

# Combine both
combined_trend = pd.concat([historical_yearly, forecast_yearly], ignore_index=True)

fig2 = go.Figure()

# Historical line
hist_data = combined_trend[combined_trend['TYPE'] == 'Historical']
fig2.add_trace(go.Scatter(
    x=hist_data['YEAR'],
    y=hist_data['SEVERITY'],
    name='Historical (2000-2024)',
    mode='lines+markers',
    line=dict(color='#4ECDC4', width=3),
    marker=dict(size=6),
))

# Forecast line
fcst_data = combined_trend[combined_trend['TYPE'] == 'Forecast']
fig2.add_trace(go.Scatter(
    x=fcst_data['YEAR'],
    y=fcst_data['SEVERITY'],
    name='Forecast (2025-2029)',
    mode='lines+markers',
    line=dict(color='#FFE66D', width=3, dash='dash'),
    marker=dict(size=6, symbol='diamond'),
))

fig2.update_layout(
    title='<b>Severity: Historical vs Forecast</b><br><sub>Comparing past trends with predictions for future years</sub>',
    xaxis_title='Year',
    yaxis_title='Average Severity Index',
    template='plotly_white',
    height=500,
    hovermode='x unified',
    font=dict(size=12)
)

fig2.show()


In [ ]:
# Chart 3: Top 15 States by Predicted Severity (2025-2029 Average)

top_states_df = forecast_summary.groupby('STATE')['SEVERITY_FORECAST'].mean().nlargest(15).reset_index()
top_states_df = top_states_df.sort_values('SEVERITY_FORECAST')

fig3 = go.Figure()

fig3.add_trace(go.Bar(
    y=top_states_df['STATE'],
    x=top_states_df['SEVERITY_FORECAST'],
    orientation='h',
    marker=dict(
        color=top_states_df['SEVERITY_FORECAST'],
        colorscale='Reds',
        showscale=True,
        colorbar=dict(title='Severity')
    ),
    text=top_states_df['SEVERITY_FORECAST'].round(3),
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Predicted Severity: %{x:.3f}<extra></extra>'
))

fig3.update_layout(
    title='<b>Top 15 States: Predicted Severity (2025-2029)</b><br><sub>States at highest risk from extreme weather events</sub>',
    xaxis_title='Average Predicted Severity Index',
    yaxis_title='State',
    template='plotly_white',
    height=600,
    showlegend=False,
    font=dict(size=12)
)

fig3.show()


In [ ]:
# Chart 4: Event Type Distribution (By Frequency and Severity)

event_analysis = df.groupby('EVENT_TYPE').agg({
    'SEVERITY_INDEX': ['mean', 'sum'],
    'YEAR': 'count'  # Count of events
}).reset_index()

event_analysis.columns = ['EVENT_TYPE', 'AVG_SEVERITY', 'TOTAL_SEVERITY', 'EVENT_COUNT']
event_analysis = event_analysis.nlargest(12, 'EVENT_COUNT')

fig4 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Event Count by Type', 'Average Severity by Type'),
    specs=[[{'type':'bar'}, {'type':'bar'}]]
)

# Left: Event count
fig4.add_trace(
    go.Bar(
        x=event_analysis['EVENT_TYPE'],
        y=event_analysis['EVENT_COUNT'],
        name='Event Count',
        marker=dict(color='#4ECDC4'),
        hovertemplate='<b>%{x}</b><br>Events: %{y}<extra></extra>'
    ),
    row=1, col=1
)

# Right: Average severity
fig4.add_trace(
    go.Bar(
        x=event_analysis['EVENT_TYPE'],
        y=event_analysis['AVG_SEVERITY'],
        name='Average Severity',
        marker=dict(color='#FF6B6B'),
        hovertemplate='<b>%{x}</b><br>Avg Severity: %{y:.3f}<extra></extra>'
    ),
    row=1, col=2
)

fig4.update_xaxes(tickangle=45, row=1, col=1)
fig4.update_xaxes(tickangle=45, row=1, col=2)
fig4.update_yaxes(title_text='Count', row=1, col=1)
fig4.update_yaxes(title_text='Severity Index', row=1, col=2)

fig4.update_layout(
    title_text='<b>Event Types: Frequency vs Severity</b><br><sub>Which events are most common? Which are most damaging?</sub>',
    height=500,
    showlegend=False,
    template='plotly_white',
    font=dict(size=11)
)

fig4.show()


In [ ]:
# Chart 5: Seasonal Pattern Heatmap (Top 10 States × Months)

# Get top 10 states by historical severity
top_10_states = df.groupby('STATE')['SEVERITY_INDEX'].mean().nlargest(10).index.tolist()

# Create severity by state and month for top 10 states
seasonal_data = df[df['STATE'].isin(top_10_states)].groupby(['STATE', 'MONTH']).agg({
    'SEVERITY_INDEX': 'mean'
}).reset_index()

# Pivot for heatmap format
seasonal_pivot = seasonal_data.pivot(index='STATE', columns='MONTH', values='SEVERITY_INDEX')

# Create heatmap
fig5 = go.Figure(data=go.Heatmap(
    z=seasonal_pivot.values,
    x=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
    y=seasonal_pivot.index,
    colorscale='Reds',
    colorbar=dict(title='Severity'),
    hovertemplate='<b>%{y}</b> - %{x}<br>Severity: %{z:.3f}<extra></extra>'
))

fig5.update_layout(
    title='<b>Seasonal Pattern Heatmap</b><br><sub>Which states are vulnerable in which months? (Top 10 states)</sub>',
    xaxis_title='Month',
    yaxis_title='State',
    height=500,
    template='plotly_white',
    font=dict(size=12)
)

fig5.show()


In [ ]:
print("\n" + "="*70)
print("VISUALIZATIONS COMPLETE")
print("="*70)
print("\n5 Charts Created:")
print("  1️⃣  Historical Trend (2000-2024) - Shows if events getting worse")
print("  2️⃣  Historical vs Forecast - Compare past patterns with future")
print("  3️⃣  Top 15 States - Geographic risk distribution")
print("  4️⃣  Event Type Analysis - What causes most damage?")
print("  5️⃣  Seasonal Heatmap - When are states most vulnerable?")
print("\nNext: Assemble these into an interactive dashboard!")
